In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr

# Change working directory to the notebook's directory
notebook_dir = os.path.dirname(os.path.abspath("frozen_linear_age.ipynb"))
os.chdir(notebook_dir)

# -------------------
# Constants for de-normalization of ages (kept for reference if needed)
# -------------------
AGE_MEAN = np.float64(466.90909090909093)
AGE_STD = np.float64(308.48141569996193)

# -------------------
# Config: folds and file patterns
# -------------------
n_folds = 5
train_pattern = "../../splits/hcp/train_subject_list_agdev_{}"  # ..._0 ..._4
val_pattern   = "../../splits/hcp/val_subject_list_agdev_{}"    # ..._0 ..._4
test_pattern  = "../../splits/hcp/test_subject_list_agdev_{}"   # ..._0 ..._4

# -------------------
# Feature sets
# -------------------
feature_sets = ["../../latents/cls_hcpagdev_k8pcq4ai_300.npz",
                ]
out_dir = "cv_regression_results_agdev"
os.makedirs(out_dir, exist_ok=True)

# -------------------
# Load ages metadata (normalized ages)
# -------------------
df = pd.read_csv("../../metadata/hcpagdev_metadata.csv")
age_map = dict(zip(df["src_subject_id"].astype(str), df["normalize_age"].astype(np.float64)))

def save_predictions_csv(csv_path, subject_ids, y_true, y_pred):
    df = pd.DataFrame({
        "subject_id": subject_ids,
        "y_true": y_true.astype(float),
        "y_pred": y_pred.astype(float),
    })
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)

# -------------------
# Main loop: feature sets -> folds
# -------------------
rows = []  # accumulate per-fold results
for feat_file in feature_sets:
    features_dict = np.load(feat_file, allow_pickle=True)

    

    # Prepare plotting figure: 1 x n_folds subplots
    fig, axes = plt.subplots(1, n_folds, figsize=(4 * n_folds, 4), squeeze=False)
    axes = axes.ravel()

    for fold in range(n_folds):
        train_fname = train_pattern.format(fold)
        val_fname   = val_pattern.format(fold)
        test_fname  = test_pattern.format(fold)  # not used here

        # Check files exist
        if not os.path.exists(train_fname) or not os.path.exists(val_fname):
            print(f"Warning: missing file(s) for fold {fold}: {train_fname} or {val_fname} not found. Skipping fold.")
            continue

        train_ids = np.loadtxt(train_fname, dtype=str)
        val_ids   = np.loadtxt(val_fname, dtype=str)
        test_ids  = np.loadtxt(test_fname, dtype=str)  # not used here

        # Keep only IDs that have both features and an age label (preserve order)
        tr_ok = [sid for sid in train_ids if (sid in features_dict) and (sid.split('_')[0] in age_map)]
        va_ok = [sid for sid in val_ids   if (sid in features_dict) and (sid.split('_')[0] in age_map)]
        test_ok = [sid for sid in test_ids  if (sid in features_dict) and (sid.split('_')[0] in age_map)]

        if len(tr_ok) == 0 or len(va_ok) == 0:
            print(f"Warning: fold {fold} has empty train or val after filtering. Skipping fold.")
            continue

        # Build feature matrices and targets (normalized ages)
        X_train = np.vstack([features_dict[sid] for sid in tr_ok])
        X_val   = np.vstack([features_dict[sid] for sid in va_ok])
        X_test  = np.vstack([features_dict[sid] for sid in test_ok])

        y_train_norm = np.array([age_map[sid.split('_')[0]] for sid in tr_ok], dtype=np.float64)
        y_val_norm   = np.array([age_map[sid.split('_')[0]] for sid in va_ok], dtype=np.float64)
        y_test_norm  = np.array([age_map[sid.split('_')[0]] for sid in test_ok], dtype=np.float64)

        # -------------------
        # Normalize features based on training set (per-column)
        # -------------------
        X_mean = X_train.mean(axis=0, keepdims=True)
        X_std  = X_train.std(axis=0, keepdims=True) + 1e-8
        X_train_norm = (X_train - X_mean) / X_std
        X_val_norm   = (X_val   - X_mean) / X_std
        X_test_norm  = (X_test  - X_mean) / X_std

        # -------------------
        # Train linear regression
        # -------------------
        lr = LinearRegression()
        lr.fit(X_train_norm, y_train_norm)

        # Predict (normalized ages)
        y_pred_norm = lr.predict(X_val_norm)
        y_pred_norm_test = lr.predict(X_test_norm)

        # Metrics on normalized scale (same as your original script)
        mse = mean_squared_error(y_val_norm, y_pred_norm)
        r2  = r2_score(y_val_norm, y_pred_norm)
        try:
            rho, pval = pearsonr(y_val_norm, y_pred_norm)
        except Exception:
            rho, pval = np.nan, np.nan

        rows.append({
            "feature_file": os.path.basename(feat_file),
            "fold": fold,
            "n_train": len(y_train_norm),
            "n_val": len(y_val_norm),
            "MSE_norm": mse,
            "R2_norm": r2,
            "rho_norm": rho,
            "rho_pval": pval
        })

        csv_path = f'CV_age_results/frozen_linear_val_{fold}.csv'
        save_predictions_csv(csv_path, va_ok, y_val_norm, y_pred_norm)
        csv_path = f'CV_age_results/frozen_linear_test_{fold}.csv'
        save_predictions_csv(csv_path, test_ok, y_test_norm, y_pred_norm_test)

        # Plot for this fold
        ax = axes[fold]
        ax.scatter(y_val_norm, y_pred_norm, alpha=0.7)
        ax.plot([y_val_norm.min(), y_val_norm.max()], [y_val_norm.min(), y_val_norm.max()], 'r--')
        ax.set_xlabel("Ground Truth (norm age)")
        ax.set_ylabel("Predicted (norm age)")
        ax.set_title(f"fold {fold}\nMSE={mse:.3f}, ρ={np.nan_to_num(rho):.3f}")

# End folds loop

# Save per-fold results to CSV
df_res = pd.DataFrame(rows)
csv_out = os.path.join(out_dir, f"cv_frozen_regression_table_{os.path.basename(feat_file)}.csv")
df_res.to_csv(csv_out, index=False)
print(f"\nPer-fold results for {feat_file}:")
print(df_res)

# Finalize and show plot
fig.suptitle(f"Age regression predictions per fold — {os.path.basename(feat_file)}", fontsize=12)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("Done. Per-fold CSVs and summaries (when available) saved to:", out_dir)





In [ ]:
# Aggregate df_res by feature_file and compute mean ± std of rho_norm
if 'df_res' in globals() and (isinstance(df_res, (pd.DataFrame)) and not df_res.empty):
    agg = df_res.groupby('feature_file')['rho_norm'].agg(['mean','std']).reset_index()
    agg['rho_mean_std'] = agg.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1)
    agg = agg.rename(columns={'mean': 'rho_mean', 'std': 'rho_std'})[['feature_file','rho_mean','rho_std','rho_mean_std']]
    print('Mean ± std of rho_norm by feature_file:')
    display(agg[['feature_file','rho_mean_std']])
    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, 'rho_norm_summary_by_feature.csv')
    agg.to_csv(out_csv, index=False)
    print(f'Saved summary CSV: {out_csv}')
else:
    print('df_res not found or empty; run the previous cells to compute df_res before aggregating.')